# Caso 01 — IA e ciência de dados na saúde

**Tema:** Aprendizado de Máquina Clássico (Árvore de Decisão, Naive Bayes e KNN)  
**Dataset:** Pima Indians Diabetes (UCI)  
**Objetivo clínico:** minimizar falsos negativos (maximizar **Recall**), pois o custo de não diagnosticar um paciente diabético é crítico em telemedicina.

Este notebook implementa o experimento comparativo pedido no enunciado. As figuras são exportadas para a pasta `figuras/`.

## 1. Setup e carga dos dados

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

BASE_DIR = Path(".")
DATA_PATH = BASE_DIR / "dados" / "diabetes.csv"
FIG_DIR = BASE_DIR / "figuras"
RESULT_DIR = BASE_DIR / "resultados"
FIG_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 10
TEST_SIZE = 0.20

ZERO_AS_MISSING = [
    "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"
]

SCORING = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight"})

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
display(df.head())
print("\nDistribuição de Outcome:")
print(df["Outcome"].value_counts(normalize=True).rename("proporção").round(3))

## 2. Preparação dos dados

No Pima Indians Diabetes, zeros em Glucose, BloodPressure, SkinThickness, Insulin e BMI são **valores ausentes** (não fisiológicos).  
A imputação (mediana) e o Min-Max do KNN entram em `Pipeline` para **evitar vazamento de informação** na validação cruzada.

In [ ]:
df_prep = df.copy()
df_prep[ZERO_AS_MISSING] = df_prep[ZERO_AS_MISSING].replace(0, np.nan)

print("Ausentes após tratar zeros:")
print(df_prep[ZERO_AS_MISSING].isna().sum())

X = df_prep.drop(columns=["Outcome"])
y = df_prep["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"\nHold-out estratificado: treino={len(X_train)} | teste={len(X_test)}")

## 3. Metodologia de validação — Stratified K-Fold (K=10)

**Por que K=10?** Com 768 amostras, K=10 é o padrão empírico: cada fold de teste tem ~77 amostras, o que estabiliza a estimativa sem tornar o treino excessivamente pequeno (como em leave-one-out).

**Por que estratificado?** A classe positiva (~35%) é minoritária; a estratificação preserva a proporção em cada fold.

A validação cruzada reduz o viés de um único particionamento e permite estimar a **variância** do erro de generalização (média ± desvio entre folds).

## 4. Engenharia e seleção de modelos

- **Árvore de Decisão:** Gini ou Entropia escolhem o atributo que mais reduz impureza; `max_depth` e `min_samples_leaf` atuam como pós-poda / regularização contra overfitting.
- **Naive Bayes Gaussiano:** adequado a atributos contínuos (ao contrário do Multinomial, voltado a contagens). A independência condicional é uma simplificação — em dados clínicos costuma ser violada, mas o classificador ainda pode funcionar bem na prática.
- **KNN:** sensível à escala → **Min-Max**. O hiperparâmetro K e a métrica (Euclidiana vs Manhattan) moldam a fronteira e a robustez a ruído.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

grids = {
    "Árvore de Decisão": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]),
        {
            "clf__criterion": ["gini", "entropy"],
            "clf__max_depth": [3, 4, 5, 6, 8, None],
            "clf__min_samples_leaf": [5, 10, 15, 20],
            "clf__min_samples_split": [2, 10, 20],
        },
    ),
    "Naive Bayes (Gaussiano)": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", GaussianNB()),
        ]),
        {"clf__var_smoothing": np.logspace(-11, -7, 5)},
    ),
    "KNN": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler()),
            ("clf", KNeighborsClassifier()),
        ]),
        {
            "clf__n_neighbors": list(range(3, 22, 2)),
            "clf__metric": ["euclidean", "manhattan"],
            "clf__weights": ["uniform", "distance"],
        },
    ),
}

melhores = {}
for nome, (pipe, param_grid) in grids.items():
    print(f"\n>>> GridSearchCV — {nome} (scoring=recall)")
    search = GridSearchCV(
        pipe, param_grid=param_grid, scoring="recall", cv=cv, n_jobs=-1, refit=True
    )
    search.fit(X_train, y_train)
    melhores[nome] = search.best_estimator_
    print("    Params:", search.best_params_)
    print(f"    Melhor Recall (CV no treino): {search.best_score_:.4f}")

## 5. Comparativo com 10-Fold no dataset completo

In [ ]:
linhas = []
for nome, pipe in melhores.items():
    scores = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    linha = {"modelo": nome}
    for m in SCORING:
        linha[f"{m}_media"] = float(np.mean(scores[f"test_{m}"]))
        linha[f"{m}_desvio"] = float(np.std(scores[f"test_{m}"]))
    linhas.append(linha)
    print(
        f"{nome:28s} | R={linha['recall_media']:.3f}±{linha['recall_desvio']:.3f} "
        f"| F1={linha['f1_media']:.3f}±{linha['f1_desvio']:.3f} "
        f"| AUC={linha['roc_auc_media']:.3f}±{linha['roc_auc_desvio']:.3f}"
    )

metricas_cv = pd.DataFrame(linhas)
display(metricas_cv.round(3))
metricas_cv.to_csv(RESULT_DIR / "metricas_cv_otimizados.csv", index=False)

In [ ]:
mets = ["recall", "precision", "f1", "roc_auc", "accuracy"]
labels = ["Recall", "Precision", "F1-Score", "AUC-ROC", "Acurácia"]
x = np.arange(len(mets))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5.5))
for i, (_, row) in enumerate(metricas_cv.iterrows()):
    medias = [row[f"{m}_media"] for m in mets]
    desvios = [row[f"{m}_desvio"] for m in mets]
    ax.bar(x + i * width, medias, width, yerr=desvios, capsize=4, label=row["modelo"])

ax.set_xticks(x + width)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score (média ± desvio no 10-Fold)")
ax.set_title("Comparativo dos modelos — CV estratificada (K=10)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_comparativo_cv.png")
plt.show()

## 6. Avaliação de performance (hold-out)

Além da acurácia, reportamos **Precision, Recall, F1, Matriz de Confusão, Curva ROC e AUC**.  
Neste contexto clínico, o **Recall** é prioritário: mede a fração de diabéticos corretamente identificados (1 − taxa de falsos negativos).

In [ ]:
resultados = {}
linhas_h = []

for nome, modelo in melhores.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_score = modelo.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr, tpr, _ = roc_curve(y_test, y_score)
    res = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score),
        "cm": cm, "fpr": fpr, "tpr": tpr,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "y_pred": y_pred,
    }
    resultados[nome] = res
    linhas_h.append({
        "modelo": nome,
        **{k: res[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "tn", "fp", "fn", "tp"]},
    })
    print(
        f"{nome:28s} | P={res['precision']:.3f} R={res['recall']:.3f} "
        f"F1={res['f1']:.3f} AUC={res['roc_auc']:.3f} | FN={fn} FP={fp}"
    )

pd.DataFrame(linhas_h).to_csv(RESULT_DIR / "metricas_holdout.csv", index=False)

# Melhor modelo: prioriza Recall, depois F1 e AUC (CV)
melhor_nome = (
    metricas_cv.sort_values(
        by=["recall_media", "f1_media", "roc_auc_media"], ascending=False
    ).iloc[0]["modelo"]
)
print(f"\n>>> Modelo recomendado: {melhor_nome}")
melhor = resultados[melhor_nome]
print(classification_report(
    y_test, melhor["y_pred"],
    target_names=["Não diabético", "Diabético"], digits=3
))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=melhor["cm"],
    display_labels=["Não diabético (0)", "Diabético (1)"],
).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"Matriz de Confusão — {melhor_nome}\n(conjunto de teste)")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_matriz_confusao_melhor.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(melhor["fpr"], melhor["tpr"], lw=2,
        label=f"{melhor_nome} (AUC = {melhor['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Aleatório (AUC = 0.50)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Taxa de Falsos Positivos (1 − Especificidade)")
ax.set_ylabel("Taxa de Verdadeiros Positivos (Recall)")
ax.set_title(f"Curva ROC — {melhor_nome}")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_curva_roc_melhor.png")
plt.show()

print(
    "Interpretação prática (diretoria): uma AUC de 0.85 significa que, "
    "em cerca de 85% dos pares (um paciente doente e um saudável) escolhidos "
    "ao acaso, o modelo atribui escore maior ao paciente doente — ou seja, "
    "há boa capacidade de ranqueamento diagnóstico."
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
for nome, res in resultados.items():
    ax.plot(res["fpr"], res["tpr"], lw=2,
            label=f"{nome} (AUC = {res['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Aleatório (AUC = 0.50)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Taxa de Falsos Positivos (1 − Especificidade)")
ax.set_ylabel("Taxa de Verdadeiros Positivos (Recall)")
ax.set_title("Curvas ROC — comparação dos três modelos")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "05_curvas_roc_todos.png")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (nome, res) in zip(axes, resultados.items()):
    sns.heatmap(res["cm"], annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["0", "1"], yticklabels=["0", "1"])
    ax.set_title(nome)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
fig.suptitle("Matrizes de Confusão — conjunto de teste", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "06_matrizes_confusao_todos.png")
plt.show()

## 7. Conclusão (rascunho para o Resumo Expandido)

Com base nos resultados (CV + hold-out), recomendamos o modelo com **melhor Recall** (e bom equilíbrio F1/AUC), alinhado à exigência do Dr. Arnaldo de minimizar falsos negativos.

**Trade-off interpretabilidade × desempenho:**
- A **Árvore de Decisão** oferece regras transparentes (úteis em saúde/explicabilidade).
- **Naive Bayes** e **KNN** podem competir em métricas, mas com menos transparência clínica imediata.
- A escolha final deve ponderar Recall (segurança diagnóstica), AUC (capacidade de ranqueamento) e interpretabilidade exigida no setor de saúde.

> Os números exatos (Recall, AUC, FN/FP) aparecem nas células acima e nos CSVs em `resultados/` — use-os ao redigirmos o documento de entrega.